## Transform Races Data
1. Read bronze races table
2. Keep only the columns required for analytics(Drop url Column)
3. Standardise column names using snake_case (raceName -> race_name, circuitId -> circuits_id)
4. Rename columns to make them more meaningful (date -> race_date)
5. Remove duplicate records
6. Transform values of columns race_name to Title Case
7. Write the transformed data to silver races table

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/03.silver-helpers

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.races"
silver_table = f"{catalog_name}.{silver_schema}.races"

#### 1. Read bronze circuits table

In [0]:
races_df = (
    spark.read.table(bronze_table)
    .filter(F.col("batch_id") == v_batch_id))

#### 2. Keep only the columns required for analytics(Drop url Column)

In [0]:
from pyspark.sql import functions as F
races_selected_df = races_df.select(
    F.col("season"),
    F.col("round"),
    F.col("url"),
    F.col("raceName"),
    F.col("date"),
    F.col("circuitId"),
    F.col("ingestion_timestamp"),
    F.col("source_file"),
    F.col("batch_id")
)

#### Step 3 & 4 - Standardise column names
 - Standardise column names using snake_case (raceName -> race_name, circuitId -> circuits_id)
 - Rename columns to make them more meaningful (date -> race_date)

In [0]:
races_renamed_df = (
    races_selected_df
    .withColumnsRenamed({
        "raceName":"race_name",
        "circuitId": "circuit_id",
        "date": "race_date"
    })
)

#### Step 5 - Remove duplicate records

In [0]:
races_distinct_df = races_renamed_df.dropDuplicates(["season","round"])

In [0]:
display(races_distinct_df)

#### Step 6 - Transform values of columns race_name to Title Case

In [0]:
races_final_df = (
    races_distinct_df
    .withColumn("race_name",F.initcap(F.col("race_name")))
)

#### Step 7 -  Write the transformed data to silver circuits table

In [0]:
merge_condition="tgt.season = src.season AND tgt.round = src.round"
columns_to_update=["race_name","race_date","circuit_id", "ingestion_timestamp", "source_file","batch_id"]

write_to_silver(races_final_df,silver_table,merge_condition,columns_to_update)